In [ ]:
## Importing Libraries

import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.models import efficientnet_b0
from torchvision.models import EfficientNet_B0_Weights
print("Libraries imported.")


In [ ]:
## Setting the GPU for training model

device = torch.device("cuda")
print("Device: ", device)

if torch.cuda.is_available():
    print("GPU: ", torch.cuda.get_device_name(0))
    print("CUDA Version: ", torch.version.cuda)
else:
    print("CUDA NA")

In [ ]:
## Setting paths for training, testing and validation dataset

train_dir = r""
test_dir = r""
val_dir = r""

print("Train Path: ", train_dir)
print("Test Path: ", test_dir)
print("Validation Path: ", val_dir)

In [ ]:
## Image Transformations

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness = 0.2, contrast = 0.2, saturation = 0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])

In [ ]:
## Loading Dataset

train_dataset = datasets.ImageFolder(root = train_dir, transform = train_transforms)
test_dataset = datasets.ImageFolder(root = test_dir, transform = val_test_transforms)
val_dataset = datasets.ImageFolder(root = val_dir, transform = val_test_transforms)

print("Training Images: ", len(train_dataset))
print("Testing Images: ", len(test_dataset))
print("Validation Images: ", len(val_dataset))

print("\n Classes: ")
print(train_dataset.classes)

In [ ]:
## Creating DataLoaders

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True, num_workers = 4, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False, num_workers = 4, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle = False, num_workers = 4, pin_memory = True)

print("DataLoaders Created.")

In [ ]:
## Checking Sample Images

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize = (12, 6))

for i, ax in enumerate(axes.flatten()):
    img = images[i].permute(1, 2, 0).numpy()
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)

    ax.imshow(img)
    ax.set_title(train_dataset.classes[labels[i]])
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
## Loading EfficientNet-B0

weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights = weights)

print("EfficientNet-B0 Loaded Successfully.")

In [ ]:
## Final Classification Layer

num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, len(train_dataset.classes))
print(model.classifier)

In [ ]:
## Moving Model to GPU

model = model.to(device)
print("Model Moved To: ", device)

## Applying Loss Function

criterion = nn.CrossEntropyLoss()
print("Loss Function Ready")

## Applying Optimizer

optimizer = optim.AdamW(model.parameters(), lr = 1e-4, weight_decay = 1e-4)
print("Optimizer Ready.")

In [ ]:
## Scheduling Learning Rate

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode = 'min', factor = 0.5,  patience = 2)
print("Scheduler Ready")

In [ ]:
## Checking Parameters

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

In [ ]:
## Creating Variables for Progress Check

epochs = 15
best_val_accuracy = 0.0

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

In [ ]:
## Creating Training Function

def training_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100*correct/total
    
    return epoch_loss, epoch_acc

In [ ]:
## Creating Validation Function

def validate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)       
      
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100*correct/total
    
    return epoch_loss, epoch_acc

In [ ]:
## Training Loop

for epoch in range(epochs):
    train_loss, train_acc = training_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%")
    
    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc

        torch.save(model.state_dict(), "Visual_Model.pth")
        print("Model Saved.")

In [ ]:
## Loading Model

model.load_state_dict(torch.load("Visual_Model.pth"))
model.eval()

print("Best Model Loaded")

In [ ]:
## Creating Evaluations

from sklearn.metrics import accuracy_score

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = accuracy_score(all_labels, all_preds)

print(f"Test Accuracy: {test_accuracy*100:.2f}%")

In [ ]:
## Classification Report

from sklearn.metrics import classification_report

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=train_dataset.classes
    )
)

In [ ]:
## Confusion Matrix

from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=train_dataset.classes,
    yticklabels=train_dataset.classes
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=train_dataset.classes,
    yticklabels=train_dataset.classes
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.savefig(
    "Visual_Confusion_Matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(val_accuracies, label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training vs Validation Accuracy")

plt.legend()

plt.show()